# Setup Network Outbound Table

In [0]:
%pip install {dbutils.widgets.get("lib_path") + "/.internal/*.whl"}
%restart_python

In [0]:
from sentinel_helpers.log_analytics import *

## Environment

In [0]:
dbutils.widgets.text("tenant_id", "")
dbutils.widgets.text("subscription_id", "")
dbutils.widgets.text("sp_client_id", "")
dbutils.widgets.text("sp_client_secret_scope", "")
dbutils.widgets.text("sp_client_secret_key", "")
dbutils.widgets.text("resource_group_name", "")
dbutils.widgets.text("location", "")
dbutils.widgets.text("log_analytics_workspace_name", "")
dbutils.widgets.text("log_analytics_resource_group_name", "")
dbutils.widgets.text("table_name", "AdbSystemAccessOutboundNetwork")

In [0]:
tenant_id = dbutils.widgets.get("tenant_id")
subscription_id = dbutils.widgets.get("subscription_id")
sp_client_id = dbutils.widgets.get("sp_client_id")
sp_client_secret_scope = dbutils.widgets.get("sp_client_secret_scope")
sp_client_secret_key = dbutils.widgets.get("sp_client_secret_key")
resource_group_name = dbutils.widgets.get("resource_group_name")
location = dbutils.widgets.get("location")
log_analytics_workspace_name = dbutils.widgets.get("log_analytics_workspace_name")
log_analytics_resource_group_name = dbutils.widgets.get("log_analytics_resource_group_name")
table_name = dbutils.widgets.get("table_name")

In [ ]:
sp_secret = dbutils.secrets.get(sp_client_secret_scope, sp_client_secret_key)

In [ ]:
from azure.identity import ClientSecretCredential

credentials = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=sp_client_id,
    client_secret=sp_secret,
)

## Required roles on the Service Principal
- **Log Analytics Contributor** and **Monitoring Contributor**: to create tables, DCEs and DCRs
- **Monitoring Metrics Publisher**: to publish logs to the DCEs

## Azure Resources Setup

In [ ]:
log_analytics_table_schema = Schema(
    name=f"{table_name}_CL",
    description="Databricks system.access.outbound_network system table",
    columns=[
        Column(name="TimeGenerated", type=ColumnTypeEnum.DATE_TIME),
        Column(name="TimeIngested", type=ColumnTypeEnum.DATE_TIME),
        Column(name="account_id", type=ColumnTypeEnum.STRING),
        Column(name="workspace_id", type=ColumnTypeEnum.STRING),
        Column(name="event_id", type=ColumnTypeEnum.STRING),
        Column(name="destination_type", type=ColumnTypeEnum.STRING),
        Column(name="destination", type=ColumnTypeEnum.STRING),
        Column(name="dns_event", type=ColumnTypeEnum.DYNAMIC),
        Column(name="ip_event", type=ColumnTypeEnum.DYNAMIC),
        Column(name="storage_event", type=ColumnTypeEnum.DYNAMIC),
        Column(name="event_time", type=ColumnTypeEnum.DATE_TIME),
        Column(name="access_type", type=ColumnTypeEnum.STRING),
    ],
)

In [ ]:
ensure_azure_resources(
    credentials=credentials,
    subscription_id=subscription_id,
    resource_group_name=resource_group_name,
    location=location,
    log_analytics_resource_group_name=log_analytics_resource_group_name,
    log_analytics_workspace_name=log_analytics_workspace_name,
    table_name=table_name,
    table_schema=log_analytics_table_schema,
)